In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

In [ ]:

import ast
# Load data
task_table = pd.read_csv('data/task_setup.csv')

# calculate average isi
print(task_table['extracted_isi'])
task_table['avg_isi'] = task_table['extracted_isi'].apply(lambda x: np.mean(ast.literal_eval(x)) if str(x).strip().lower() != "nan" else np.nan)

# Set visualization style
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 1, figsize=(12, 12))

# Plot isi histogram
isi_data = task_table['avg_isi'].dropna()

# Create bins dynamically in 100ms steps up to the maximum ISI value
sns.histplot(
    isi_data, 
    bins=range(0, int(isi_data.max()) + 200, 100), 
    ax=axes[0], 
    color='skyblue', 
    edgecolor='black'
)

axes[0].set_title('Distribution of Average Inter-Stimulus Intervals (100ms Bins)', fontsize=14)
axes[0].set_ylabel('Number of Studies', fontsize=12)
axes[0].set_xlabel('ISI (ms)', fontsize=12)

# Set tick marks every 500ms with minor ticks at 100ms intervals
axes[0].xaxis.set_major_locator(ticker.MultipleLocator(500))
axes[0].xaxis.set_minor_locator(ticker.MultipleLocator(100))

# Box plot for nr trials per paradigm
# Map exact string values in the dataset to desired display labels
task_table['paradigm_label'] = task_table['paradigm'].replace({
    'CPT': 'Continuous Performance Test (CPT)',
    'Working Memory / Change Detection': 'Working Memory / Change Detection Tasks'
})

plot_order = [
    'GoNoGo',
    'Continuous Performance Test (CPT)',
    'Stop Signal Task (SST)',
    'N-Back Task',
    'Working Memory / Change Detection Tasks'
]

# Draw the underlying box plot
sns.boxplot(
    data=task_table, 
    y='paradigm_label', 
    x='nr_trials', 
    order=plot_order, 
    color='lightgray', 
    ax=axes[1],
    fliersize=0
)

# Overlay the individual study points (stripplot)
sns.stripplot(
    data=task_table, 
    y='paradigm_label', 
    x='nr_trials', 
    order=plot_order, 
    alpha=0.7, 
    jitter=True, 
    size=6, 
    ax=axes[1]
)

axes[1].set_title('Number of Trials across Paradigms', fontsize=14)
axes[1].set_xlabel('Nr trials', fontsize=12)
axes[1].set_ylabel('', fontsize=12)


plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('plots/isi_100ms_and_trials_plot.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

eeg_setup = pd.read_csv('data/eeg_setup.csv')


STEP_MAP = {
    1: 'EOG Recording',
    2: 'High-pass Filter',
    3: 'Line Noise Filter',
    4: 'Bad Channel Rejection',
    5: 'Channel Interpolation',
    6: 'Re-referencing',
    7: 'ICA',
    8: 'Epoching',
    9: 'Artifact Rejection',
    10: 'Manual Selection',
    11: 'Manual Rejection'
}

# --- 2. Extract preprocessing steps into a long-format table ---
# Values such as "01_02_06" become the numeric steps [1, 2, 6].
steps_long = (
    eeg_setup[['short_id', 'eeg_preprocessing_steps']]
    .assign(step=lambda data: data['eeg_preprocessing_steps'].str.split('_'))
    .explode('step')
    .dropna(subset=['step'])
    .assign(step=lambda data: pd.to_numeric(data['step'], errors='coerce'))
    .dropna(subset=['step'])
    .assign(step=lambda data: data['step'].astype(int))
    .query('step in @STEP_MAP')
)

# Create presence matrix
plot_data = (
    steps_long.assign(present=1)
    .pivot_table(index='step', columns='short_id', values='present', aggfunc='max', fill_value=0)
    .reindex(index=STEP_MAP.keys(), fill_value=0)
)
plot_data.index = [STEP_MAP[step] for step in plot_data.index]
plot_data = plot_data.reindex(sorted(plot_data.columns), axis=1)

step_counts_per_study = plot_data.sum(axis=0)

# --- 4. Plotting ---
fig, (ax_bar, ax_mat) = plt.subplots(
    nrows=2, 
    ncols=1, 
    figsize=(12, 8), 
    gridspec_kw={'height_ratios': [1, 3], 'hspace': 0.08},
    sharex=True
)

# Top Bar Chart: Total steps used per study
bars = ax_bar.bar(
    range(len(step_counts_per_study)), 
    step_counts_per_study, 
    color='#34495e', 
    width=0.55
)
ax_bar.set_ylabel("Total Steps\nUsed", fontsize=10, fontweight='bold')
ax_bar.set_ylim(0, 13)
ax_bar.set_yticks(np.arange(0, 14, 2))
ax_bar.grid(axis='y', linestyle='--', alpha=0.5)
ax_bar.spines[['top', 'right']].set_visible(False)

# Add exact count values on top of bars
for bar in bars:
    height = bar.get_height()
    ax_bar.annotate(
        f'{int(height)}',
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha='center', va='bottom',
        fontsize=10, fontweight='bold'
    )

# Bottom Presence Matrix: Connected dots per preprocessing step
n_steps, n_studies = plot_data.shape

# Light background grid lines
for y in range(n_steps):
    ax_mat.axhline(y, color='#ecf0f1', zorder=1, lw=1.5)

for x in range(n_studies):
    active_y = np.where(plot_data.iloc[:, x] == 1)[0]
    inactive_y = np.where(plot_data.iloc[:, x] == 0)[0]
    
    # Vertical connection line across active steps
    if len(active_y) > 1:
        ax_mat.plot(
            [x, x], [active_y.min(), active_y.max()], 
            color='#2980b9', lw=2.5, zorder=2
        )
    
    # Active dots
    ax_mat.scatter(
        [x] * len(active_y), active_y, 
        color='#2980b9', s=110, zorder=3
    )
    # Inactive dots
    ax_mat.scatter(
        [x] * len(inactive_y), inactive_y, 
        color='#bdc3c7', s=35, alpha=0.6, zorder=2
    )

# Formatting Bottom Matrix Axes
ax_mat.set_yticks(range(n_steps))
ax_mat.set_yticklabels(plot_data.index, fontsize=10)
ax_mat.set_xticks(range(n_studies))
ax_mat.set_xticklabels(plot_data.columns, rotation=45, ha='right', fontsize=10, fontweight='bold')
ax_mat.set_ylim(-0.5, n_steps - 0.5)
ax_mat.invert_yaxis()  # Puts step 1 at the top, step 12 at the bottom
ax_mat.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax_mat.tick_params(left=False, bottom=False)

plt.tight_layout()
plt.savefig('plots/preprocessing_steps.png', bbox_inches='tight', pad_inches=0.2)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
import matplotlib.ticker as ticker

df = pd.read_csv('data/eeg_setup.csv')

# Binning 256->250 and 512->500
def parse_freq(val):
    if pd.isna(val): return None
    nums = re.findall(r'\d+', str(val))
    if nums:
        num = int(nums[-1])
        if num == 512: return 500
        if num == 256: return 250
        return num
    return None

df['freq_parsed'] = df['sampling_freq'].apply(parse_freq)

# Parsing References
def parse_ref(val):
    if pd.isna(val): return 'Unknown'
    val_lower = str(val).lower()
    if 'average' in val_lower: return 'Average Reference'
    if 'mastoid' in val_lower: return 'Linked Mastoids'
    if 'ear' in val_lower: return 'Linked Ears'
    return 'Other'

df['ref_parsed'] = df['offline reference'].apply(parse_ref)


# Plotting Setup (2x2 grid)
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot A: Frequencies
freq_counts = df['freq_parsed'].value_counts().sort_index()
sns.barplot(
    x=freq_counts.index.astype(int), 
    y=freq_counts.values.astype(int), 
    color="#ED8963", 
    ax=axes[0, 0]
)
axes[0, 0].set_title('Analysis Sampling Frequency (Hz)')
axes[0, 0].set_xlabel('Frequency (Hz)')
axes[0, 0].set_ylabel('Number of Studies')
axes[0, 0].set_yticks(range(0, 18 + 1, 2))

# Plot B: References
ref_counts = df['ref_parsed'].value_counts()
sns.barplot(
    x=ref_counts.index, 
    y=ref_counts.values, 
    palette=["#475680", "#338382", "#74B969"], 
    hue=ref_counts.index,
    legend=False,
    ax=axes[0, 1]
)
axes[0, 1].set_title('Offline Reference Choice')
axes[0, 1].set_xlabel('Reference Type')
axes[0, 1].set_ylabel('Number of Studies')

# Plot C: Epoch Boundaries (Start and End)
sns.histplot(
    df['epoch_start'].dropna(), bins=20, kde=True, 
    color="blue", alpha=0.5, edgecolor="white", label='Epoch Start', ax=axes[1, 0]
)
sns.histplot(
    df['epoch_end'].dropna(), bins=20, kde=True, 
    color="cyan", alpha=0.5, edgecolor="white", label='Epoch End', ax=axes[1, 0]
)
axes[1, 0].set_title('Consensus: Epoch Boundaries Distribution')
axes[1, 0].set_xlabel('Time (ms)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].legend()

# Plot D: Baseline Start
sns.histplot(
    df['baseline_start'].dropna(), bins=20, kde=True, 
    color="brown", alpha=0.7, edgecolor="white", label='Baseline Start', ax=axes[1, 1]
)
axes[1, 1].set_title('Consensus: Baseline Start Distribution')
axes[1, 1].set_xlabel('Time (ms)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].legend()

# Annotate Subplots (a) to (d)
for ax, label in zip(axes.flat, string.ascii_lowercase[:4]):
    ax.text(
        -0.08, 1.05, f"({label})",
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        va="bottom",
        ha="right"
    )

for ax in axes[1, :]:
    # Set tick mark positions every 100 without text labels
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(100))
    
    # Make the tick marks visible on the axis spine
    ax.tick_params(axis='x', which='minor', bottom=True, length=4, color='gray')
    
    # Ensure no grid lines are drawn for these ticks
    ax.grid(False, which='minor', axis='x')

# Final layout adjustments
plt.tight_layout()
plt.savefig('plots/eeg_setup.png', bbox_inches='tight', pad_inches=0.2)